In [23]:
import pandas as pd
import numpy as np
import gym
from gym import spaces

import torch
import torch.nn as nn
import torch.optim as optim

import matplotlib.pyplot as plt

In [24]:
# 데이터 로드 및 전처리 함수 정의
def load_stock_data(stock_path):
    stock_data = pd.read_csv(stock_path)
    stock_data['Date'] = pd.to_datetime(stock_data['Date'])
    stock_data.set_index('Date', inplace=True)
    return stock_data

In [53]:
import feedparser
from datetime import datetime, timedelta
from urllib.parse import quote

def fetch_real_time_news_google(query, from_date, to_date, max_articles=10):
    # 날짜 형식을 Google News에 맞게 변환합니다.
    # from_date_str = from_date.strftime('%Y-%m-%d')
    # to_date_str = to_date.strftime('%Y-%m-%d')
    encoded_query = quote(query)
    # 검색 쿼리와 기간을 포함한 URL을 생성합니다.
    feed_url = f"https://news.google.com/rss/search?q={encoded_query}+after:{from_date}+before:{to_date}&hl=en-US&gl=US&ceid=US:en"

    feed = feedparser.parse(feed_url)
    
    articles = []
    for entry in feed.entries[:max_articles]:
        article_dict = {
            'title': entry.title,
            'published': entry.published,
            'link': entry.link,
            'summary': entry.summary
        }
        articles.append(article_dict)
    
    return pd.DataFrame(articles)

# 예시: 'stock market' 검색어로 2023년 1월 1일부터 1월 31일까지의 구글 뉴스 기사 가져오기
query = 'stock market'
from_date = '2023-01-01'
to_date = '2023-12-31'
max_articles = 100000

news_articles = fetch_real_time_news_google(query, from_date, to_date, max_articles)
print(len(news_articles['title']))

30


In [ ]:
def calculate_moving_average(data, window):
    return data.rolling(window=window).mean()

def calculate_rsi(data, window):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).fillna(0)
    loss = (-delta.where(delta < 0, 0)).fillna(0)

    avg_gain = gain.rolling(window=window).mean()
    avg_loss = loss.rolling(window=window).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

In [32]:
class ActorCritic(nn.Module):
    def __init__(self, input_dim, action_dim):
        super(ActorCritic, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 256)
        self.fc_actor = nn.Linear(256, action_dim)
        self.fc_critic = nn.Linear(256, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        action_probs = torch.softmax(self.fc_actor(x), dim=-1)
        state_value = self.fc_critic(x)
        return action_probs, state_value

class PPOAgent:
    def __init__(self, state_dim, action_dim, lr=3e-4, gamma=0.99, clip_epsilon=0.2, K_epochs=4, update_timestep=2000):
        self.gamma = gamma
        self.clip_epsilon = clip_epsilon
        self.K_epochs = K_epochs
        self.update_timestep = update_timestep

        self.policy = ActorCritic(state_dim, action_dim).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        self.policy_old = ActorCritic(state_dim, action_dim).to(device)
        self.policy_old.load_state_dict(self.policy.state_dict())

        self.MseLoss = nn.MSELoss()

        self.memory = {
            'states': [],
            'actions': [],
            'logprobs': [],
            'rewards': [],
            'is_terminals': []
        }
        self.timestep = 0

    def act(self, state):
        state = torch.FloatTensor(state).to(device)
        action_probs, _ = self.policy_old(state)
        action_dist = torch.distributions.Categorical(action_probs)
        action = action_dist.sample()
        return action.item(), action_dist.log_prob(action).item()

    def evaluate(self, state, action):
        action_probs, state_value = self.policy(state)
        action_dist = torch.distributions.Categorical(action_probs)
        action_logprobs = action_dist.log_prob(action)
        dist_entropy = action_dist.entropy()
        return action_logprobs, torch.squeeze(state_value), dist_entropy

    def store(self, state, action, logprob, reward, is_terminal):
        self.memory['states'].append(state)
        self.memory['actions'].append(action)
        self.memory['logprobs'].append(logprob)
        self.memory['rewards'].append(reward)
        self.memory['is_terminals'].append(is_terminal)

    def update(self):
        rewards = []
        discounted_reward = 0
        for reward, is_terminal in zip(reversed(self.memory['rewards']), reversed(self.memory['is_terminals'])):
            if is_terminal:
                discounted_reward = 0
            discounted_reward = reward + self.gamma * discounted_reward
            rewards.insert(0, discounted_reward)
        
        rewards = torch.tensor(rewards).to(device)
        rewards = (rewards - rewards.mean()) / (rewards.std() + 1e-7)
        
        old_states = torch.FloatTensor(self.memory['states']).to(device)
        old_actions = torch.FloatTensor(self.memory['actions']).to(device)
        old_logprobs = torch.FloatTensor(self.memory['logprobs']).to(device)

        for _ in range(self.K_epochs):
            logprobs, state_values, dist_entropy = self.evaluate(old_states, old_actions)
            ratios = torch.exp(logprobs - old_logprobs.detach())

            advantages = rewards - state_values.detach()
            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * advantages

            loss = -torch.min(surr1, surr2) + 0.5 * self.MseLoss(state_values, rewards) - 0.01 * dist_entropy

            self.optimizer.zero_grad()
            loss.mean().backward()
            self.optimizer.step()

        self.policy_old.load_state_dict(self.policy.state_dict())
        self.memory = {
            'states': [],
            'actions': [],
            'logprobs': [],
            'rewards': [],
            'is_terminals': []
        }

In [33]:
class StockTradingEnv(gym.Env):
    def __init__(self, stock_data, indicator_data, news_data):
        super(StockTradingEnv, self).__init__()
        self.stock_data = stock_data
        self.indicator_data = indicator_data
        self.news_data = news_data
        self.current_step = 0
        self.window_size = 10
        self.inventory = []
        self.total_profit = 0
        self.done = False
        self.actions = []
        
        self.action_space = spaces.Discrete(3)  # Actions: 0 = Hold, 1 = Buy, 2 = Sell
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.window_size * 3 + 2,), dtype=np.float32)

    def reset(self):
        self.current_step = 0
        self.inventory = []
        self.total_profit = 0
        self.done = False
        self.actions = []
        return self._get_observation()

    def _get_observation(self):
        obs = np.concatenate((self.stock_data.iloc[self.current_step:self.current_step + self.window_size].values.flatten(),
                              self.indicator_data.iloc[self.current_step:self.current_step + self.window_size].values.flatten(),
                              [len(self.inventory), self.total_profit]))
        return obs

    def step(self, action):
        self.done = False
        self.actions.append(action)
        reward = 0
        if action == 1:  # Buy
            self.inventory.append(self.stock_data.iloc[self.current_step]['Close'])
        elif action == 2 and len(self.inventory) > 0:  # Sell
            bought_price = self.inventory.pop(0)
            profit = self.stock_data.iloc[self.current_step]['Close'] - bought_price
            reward = profit
            self.total_profit += profit

        self.current_step += 1
        if self.current_step >= len(self.stock_data) - self.window_size:
            self.done = True

        obs = self._get_observation()
        return obs, reward, self.done, {}

In [34]:
stock_data = load_stock_data('./datas/data.csv')
indicator_data = calculate_moving_average(stock_data['Close'], 14)
rsi_data = calculate_rsi(stock_data['Close'], 14)
indicator_data = pd.concat([indicator_data, rsi_data], axis=1).dropna()

news_data = fetch_real_time_news(api_key, query, from_date, to_date)

env = StockTradingEnv(stock_data, indicator_data, news_data)

ValueError: Error fetching news: 426

In [ ]:
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

In [20]:
device = torch.device("mps" if torch.cuda.is_available() else "cpu")
ppo_agent = PPOAgent(state_dim, action_dim)

NameError: name 'state_dim' is not defined

In [21]:
n_episodes = 1000
for episode in range(n_episodes):
    state = env.reset()
    while True:
        action, log_prob = ppo_agent.act(state)
        next_state, reward, done, _ = env.step(action)
        ppo_agent.store(state, action, log_prob, reward, done)
        state = next_state
        if done:
            break
    ppo_agent.update()
    print(f'Episode {episode+1}/{n_episodes} completed')

NameError: name 'env' is not defined

In [22]:
plt.plot(env.actions)
plt.title("Actions over time")
plt.show()

NameError: name 'env' is not defined